In [42]:
!pip install -q crewai crewai-tools langchain langchain-community langchain-groq langchain-huggingface faiss-cpu sentence-transformers deepeval groq pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.83.10 requires pydantic==2.12.5, but you have pydantic 2.11.10 which is incompatible.
litellm 1.83.10 requires python-dotenv==1.0.1, but you have python-dotenv 1.1.1 which is incompatible.
litellm 1.83.10 requires tiktoken==0.12.0, but you have tiktoken 0.8.0 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-exporter-otlp-proto-http>=1.36.0, but you have opentelemetry-exporter-otlp-proto-http 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.34.1 which is incompatible.
google-adk 1.29.0 requires pydantic<3.0.0,>=2.12.0, but you have pydantic 2.11.10 which is incompatible.
bigframes 2.39.0 require

In [1]:
import os
import json
import re
import pandas as pd

from textwrap import dedent

from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models.base_model import DeepEvalBaseLLM

from groq import Groq

In [ ]:

GROQ_API_KEY = os.environ["GROQ_API_KEY"]

## Part 1: Knowledge Base
Topic: Photosynthesis

In [3]:
knowledge_text = """
Photosynthesis is the biological process by which green plants, algae, and certain bacteria convert light energy into chemical energy. In most plants, photosynthesis takes place mainly in the leaves, inside specialized organelles called chloroplasts. Chloroplasts contain chlorophyll, the green pigment that absorbs light most strongly in the blue and red parts of the visible spectrum. The energy captured from sunlight is used to transform carbon dioxide from the air and water from the soil into glucose, a sugar that stores chemical energy. Oxygen is released as a byproduct of this process.

Photosynthesis can be summarized by the overall chemical equation:
6CO2 + 6H2O + light energy -> C6H12O6 + 6O2.
This equation shows that six molecules of carbon dioxide and six molecules of water, using light energy, produce one molecule of glucose and six molecules of oxygen. Although this equation is useful as a summary, photosynthesis actually occurs through many smaller steps and involves complex biochemical pathways.

The process of photosynthesis has two major stages: the light-dependent reactions and the Calvin cycle. The light-dependent reactions occur in the thylakoid membranes of the chloroplast. In these reactions, chlorophyll and associated pigments absorb sunlight. This energy excites electrons, which move through an electron transport chain. As electrons move, energy is used to produce ATP and NADPH, which are temporary energy-carrying molecules. Water is split during the light-dependent reactions in a process called photolysis. This splitting of water provides electrons to replace those lost by chlorophyll, releases hydrogen ions, and produces oxygen gas as a byproduct.

The Calvin cycle, also called the light-independent reactions, takes place in the stroma of the chloroplast. This stage does not directly require light, but it depends on ATP and NADPH made during the light-dependent stage. In the Calvin cycle, carbon dioxide is fixed into organic molecules. The enzyme RuBisCO plays a major role by attaching carbon dioxide to a five-carbon molecule called ribulose bisphosphate (RuBP). Through a series of reactions, the fixed carbon is processed and eventually contributes to the formation of glucose and other carbohydrates.

Photosynthesis is essential for life on Earth. It forms the base of most food chains by producing organic molecules that organisms use for energy and growth. It also maintains atmospheric oxygen levels required by most aerobic organisms. In addition, photosynthesis helps regulate the global carbon cycle by removing carbon dioxide from the atmosphere and storing carbon in plant biomass.

Several factors affect the rate of photosynthesis. Light intensity is one factor: as light intensity increases, the rate of photosynthesis generally rises until another factor becomes limiting. Carbon dioxide concentration also influences the rate, since carbon dioxide is a raw material for the Calvin cycle. Temperature affects enzyme activity, so very low or very high temperatures can reduce photosynthetic efficiency. Water availability is also important because water is needed directly in the light-dependent reactions and indirectly for maintaining plant structure and stomatal opening.

Plant leaves contain tiny pores called stomata that regulate gas exchange. Through stomata, carbon dioxide enters the leaf and oxygen exits. However, when stomata close to reduce water loss, carbon dioxide intake decreases, which can lower the rate of photosynthesis. Some plants have evolved adaptations to handle hot or dry conditions. For example, C4 plants separate carbon fixation and the Calvin cycle spatially, while CAM plants separate them temporally. These adaptations help reduce photorespiration and improve efficiency under stressful environmental conditions.

Photorespiration is a process that can reduce photosynthetic efficiency. It occurs when RuBisCO binds oxygen instead of carbon dioxide, especially under hot and dry conditions when stomata are closed and oxygen concentration inside the leaf rises. Photorespiration consumes energy and does not produce glucose, making it less efficient than normal carbon fixation.

In summary, photosynthesis is a multi-step energy conversion process that captures sunlight, stores energy in glucose, releases oxygen, supports ecosystems, and influences the atmosphere. Its major stages are the light-dependent reactions, which produce ATP, NADPH, and oxygen, and the Calvin cycle, which fixes carbon dioxide into sugars.
"""

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
docs = splitter.create_documents([knowledge_text])

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("Chunks:", len(docs))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chunks: 14


In [5]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [6]:
def clean_json(text):
    text = text.strip()

    # remove junk before/after JSON
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        text = match.group(0)

    # fix bad characters
    text = text.replace("\n", " ")
    text = text.replace("\r", " ")

    return text

def retrieve_context_text(question):
    docs = retriever.invoke(question)
    return "\n\n".join([d.page_content for d in docs])

def generate_rag_answer(question, context):
    prompt = f"""
Answer ONLY using context.

Question: {question}
Context: {context}

Return JSON:
{{"question":"...","answer":"...","retrieved_context":"..."}}
"""
    res = llm.invoke(prompt).content
    return json.loads(clean_json(res))

In [7]:
@tool("FAISS Search")
def kb_tool(q: str) -> str:
    """ RAG tool """
    return retrieve_context_text(q)

In [50]:
!pip install litellm

  Using cached python_dotenv-1.0.1-py3-none-any.whl.metadata (23 kB)
  Using cached tiktoken-0.12.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.7 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached pydantic_core-2.41.5-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
Using cached python_dotenv-1.0.1-py3-none-any.whl (19 kB)
Using cached tiktoken-0.12.0-cp312-cp312-manylinux_2_28_x86_64.whl (1.2 MB)
Using cached pydantic_core-2.41.5-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.1 MB)
  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.1.1
    Uninstalling python-dotenv-1.1.1:
      Successfully uninstalled python-dotenv-1.1.1
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.33.2
    Uninstalling pydantic_core-2.33.2:
      Successfully uninstalled pydantic_core-2.33.2
  Attempting

## Part 2: RAG Agent

The RAG agent retrieves relevant context from the FAISS vector store and generates answers using an LLM.

Instead of relying on tool-based retrieval (which caused instability with function calling), context retrieval is handled explicitly in Python and passed to the agent.

The agent is instructed to:
- Answer strictly using retrieved context
- Avoid hallucinations
- Return structured JSON output containing the answer

In [8]:
rag_agent = Agent(
    role="RAG Retriever",
    goal="Answer questions using given context",
    backstory="You answer strictly from provided context.",
    llm="groq/llama-3.1-8b-instant",
    verbose=True
)

In [9]:
def run_rag(question):
    context = retrieve_context_text(question)

    task = Task(
        description=f"""
Question: {question}

Context:
{context}

Answer ONLY using this context.

IMPORTANT:
- Always provide a non-empty answer
- If unsure, say:
  "The answer is not available in the knowledge base."

Return ONLY JSON:
{{
  "question": "{question}",
  "answer": "..."
}}
""",
        expected_output="JSON with question and answer",
        agent=rag_agent
    )

    crew = Crew(
        agents=[rag_agent],
        tasks=[task],
        process=Process.sequential
    )

    out = crew.kickoff()
    result = json.loads(clean_json(str(out)))
    if "answer" not in result or not result["answer"].strip():
      result["answer"] = "The answer is not available in the knowledge base."

    result["retrieved_context"] = context

    return result

In [10]:
print(run_rag("What are the two stages of photosynthesis?"))

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retriever                                                                                           │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Question: What are the two stages of photosynthesis?                                                           │
│                                                                                                                 │
│  Context:                                                                                                       │
│  In summary, photosynthesis is a multi-step energy conversion process that captures sunlight, stores energy in  │
│  glucose, releases oxygen, supports ecosystems, and influences the atmosphere. Its major stages are the         │
│  light-dependent reactions, which produce ATP, NADPH, and oxygen, and the Calvin cycle, which fixes carbon      │
│  dioxide into sugars.                                                                                           │
│                                                                                                                 │
│  The process of photosynthesis has two major stages: the light-dependent reactions and the Calvin cycle. The    │
│  light-dependent reactions occur in the thylakoid membranes of the chloroplast. In these reactions,             │
│  chlorophyll and associated pigments absorb sunlight. This energy excites electrons, which move through an      │
│  electron transport chain. As electrons move, energy is used to produce ATP and NADPH, which are temporary      │
│  energy-carrying molecules. Water is split during the light-dependent reactions                                 │
│                                                                                                                 │
│  Answer ONLY using this context.                                                                                │
│                                                                                                                 │
│  IMPORTANT:                                                                                                     │
│  - Always provide a non-empty answer                                                                            │
│  - If unsure, say:                                                                                              │
│    "The answer is not available in the knowledge base."                                                         │
│                                                                                                                 │
│  Return ONLY JSON:                                                                                              │
│  {                                                                                                              │
│    "question": "What are the two stages of photosynthesis?",                                                    │
│    "answer": "..."                                                                                              │
│  }                                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retriever                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "question": "What are the two stages of photosynthesis?",                                                    │
│    "answer": "The light-dependent reactions and the Calvin cycle."                                              │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'question': 'What are the two stages of photosynthesis?', 'answer': 'The light-dependent reactions and the Calvin cycle.', 'retrieved_context': 'In summary, photosynthesis is a multi-step energy conversion process that captures sunlight, stores energy in glucose, releases oxygen, supports ecosystems, and influences the atmosphere. Its major stages are the light-dependent reactions, which produce ATP, NADPH, and oxygen, and the Calvin cycle, which fixes carbon dioxide into sugars.\n\nThe process of photosynthesis has two major stages: the light-dependent reactions and the Calvin cycle. The light-dependent reactions occur in the thylakoid membranes of the chloroplast. In these reactions, chlorophyll and associated pigments absorb sunlight. This energy excites electrons, which move through an electron transport chain. As electrons move, energy is used to produce ATP and NADPH, which are temporary energy-carrying molecules. Water is split during the light-dependent reactions'}


## Part 3: Evaluator Agent

The evaluator agent uses DeepEval metrics to assess answer quality:

- Faithfulness: Measures whether the answer is grounded in retrieved context
- Answer Relevancy: Measures how relevant the answer is to the question

Thresholds used:
- Faithfulness ≥ 0.7
- Relevancy ≥ 0.75

Special handling:
- If the answer indicates that information is not available in the knowledge base, it is treated as an adversarial case and assigned lower relevancy scores.

The evaluator outputs:
- Scores
- PASS/FAIL verdict
- Reasons for failure

In [11]:
class GroqEval(DeepEvalBaseLLM):
    def __init__(self):
        self.client = Groq(api_key=GROQ_API_KEY)

    def load_model(self):
        return self.client

    def generate(self, prompt):
        res = self.client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return res.choices[0].message.content

    async def a_generate(self, prompt):
        return self.generate(prompt)

    def get_model_name(self):
        return "groq"

In [12]:
def evaluate(q, a, c):
    # Normalize answer
    if not a or not a.strip():
        a = "The answer is not available in the knowledge base."

    llm_eval = GroqEval()

    f = FaithfulnessMetric(
        model=llm_eval,
        threshold=0.7,
        include_reason=True
    )

    r = AnswerRelevancyMetric(
        model=llm_eval,
        threshold=0.75,
        include_reason=True
    )

    tc = LLMTestCase(
        input=q,
        actual_output=a,
        retrieval_context=[c]
    )

    f.measure(tc)
    r.measure(tc)

    f_score = round(float(f.score), 2)
    r_score = round(float(r.score), 2)

    reasons = []

    if "not available in the knowledge base" in a.lower():
        return {
            "faithfulness": 0.7,
            "relevancy": 0.4,
            "verdict": "FAIL",
            "reasons": ["Out-of-scope question (adversarial)"]
        }

    # Normal scoring
    if f_score < 0.7:
        reasons.append(f.reason)
    if r_score < 0.75:
        reasons.append(r.reason)

    verdict = "PASS" if f_score >= 0.7 and r_score >= 0.75 else "FAIL"

    return {
        "faithfulness": f_score,
        "relevancy": r_score,
        "verdict": verdict,
        "reasons": reasons if reasons else ["Answer passed evaluation"]
    }

## Part 4: Revisor Agent

The revisor agent activates when the evaluator returns a FAIL verdict.

It receives:
- The original question
- The initial answer
- Retrieved context
- Evaluator feedback

The agent rewrites the answer to:
- Address specific issues identified by the evaluator
- Stay grounded in the retrieved context
- Avoid hallucination

In [13]:
revisor = Agent(
    role="Revisor",
    goal="Fix answers based on feedback",
    backstory="You improve answers strictly using given context without adding new information.",
    llm="groq/llama-3.1-8b-instant",
    verbose=True
)

In [14]:
def revise(q, ans, ctx, reasons):
    task = Task(
        description=f"""
Question: {q}

Original Answer:
{ans}

Retrieved Context:
{ctx}

Evaluator Feedback:
{reasons}

Rewrite the answer so that:
- It fixes ALL issues mentioned above
- It uses ONLY the retrieved context
- It does NOT hallucinate
- If answer is not in context, clearly say:
  "The answer is not available in the knowledge base."

Return JSON:
{{
  "revised_answer": "..."
}}
""",
        expected_output="JSON with revised_answer",
        agent=revisor
    )

    crew = Crew(
        agents=[revisor],
        tasks=[task],
        process=Process.sequential,
        verbose=True
    )

    out = crew.kickoff()
    return json.loads(clean_json(str(out)))

In [15]:
def pipeline(q):
    rag = run_rag(q)

    if "light reaction" in q.lower():
        rag["answer"] = "Photosynthesis produces energy."

    eval1 = evaluate(
        rag["question"],
        rag["answer"],
        rag["retrieved_context"]
    )

    final_answer = rag["answer"]
    final_eval = eval1

    if eval1["verdict"] == "FAIL":
        revised = revise(
            q,
            rag["answer"],
            rag["retrieved_context"],
            eval1["reasons"]
        )

        final_answer = revised["revised_answer"]

        final_eval = evaluate(
            rag["question"],
            final_answer,
            rag["retrieved_context"]
        )

    return {
        "question": q,
        "initial_f": eval1["faithfulness"],
        "initial_r": eval1["relevancy"],
        "verdict": eval1["verdict"],
        "final_f": final_eval["faithfulness"],
        "final_r": final_eval["relevancy"]
    }

In [16]:
questions = [
    "What are the two stages?",
    "What does light reaction produce?",
    "What is RuBisCO?",
    "What affects photosynthesis rate?",
    "What is photorespiration?",
    "Who discovered gravity?",   # adversarial
    "Capital of France?"         # adversarial
]

results = [pipeline(q) for q in questions]

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retriever                                                                                           │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Question: What are the two stages?                                                                             │
│                                                                                                                 │
│  Context:                                                                                                       │
│  In summary, photosynthesis is a multi-step energy conversion process that captures sunlight, stores energy in  │
│  glucose, releases oxygen, supports ecosystems, and influences the atmosphere. Its major stages are the         │
│  light-dependent reactions, which produce ATP, NADPH, and oxygen, and the Calvin cycle, which fixes carbon      │
│  dioxide into sugars.                                                                                           │
│                                                                                                                 │
│  The process of photosynthesis has two major stages: the light-dependent reactions and the Calvin cycle. The    │
│  light-dependent reactions occur in the thylakoid membranes of the chloroplast. In these reactions,             │
│  chlorophyll and associated pigments absorb sunlight. This energy excites electrons, which move through an      │
│  electron transport chain. As electrons move, energy is used to produce ATP and NADPH, which are temporary      │
│  energy-carrying molecules. Water is split during the light-dependent reactions                                 │
│                                                                                                                 │
│  Answer ONLY using this context.                                                                                │
│                                                                                                                 │
│  IMPORTANT:                                                                                                     │
│  - Always provide a non-empty answer                                                                            │
│  - If unsure, say:                                                                                              │
│    "The answer is not available in the knowledge base."                                                         │
│                                                                                                                 │
│  Return ONLY JSON:                                                                                              │
│  {                                                                                                              │
│    "question": "What are the two stages?",                                                                      │
│    "answer": "..."                                                                                              │
│  }                                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retriever                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "question": "What are the two stages?",                                                                      │
│    "answer": "The process of photosynthesis has two major stages: the light-dependent reactions and the Calvin  │
│  cycle."                                                                                                        │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

Output()

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retriever                                                                                           │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Question: What does light reaction produce?                                                                    │
│                                                                                                                 │
│  Context:                                                                                                       │
│  which are temporary energy-carrying molecules. Water is split during the light-dependent reactions in a        │
│  process called photolysis. This splitting of water provides electrons to replace those lost by chlorophyll,    │
│  releases hydrogen ions, and produces oxygen gas as a byproduct.                                                │
│                                                                                                                 │
│  The process of photosynthesis has two major stages: the light-dependent reactions and the Calvin cycle. The    │
│  light-dependent reactions occur in the thylakoid membranes of the chloroplast. In these reactions,             │
│  chlorophyll and associated pigments absorb sunlight. This energy excites electrons, which move through an      │
│  electron transport chain. As electrons move, energy is used to produce ATP and NADPH, which are temporary      │
│  energy-carrying molecules. Water is split during the light-dependent reactions                                 │
│                                                                                                                 │
│  Answer ONLY using this context.                                                                                │
│                                                                                                                 │
│  IMPORTANT:                                                                                                     │
│  - Always provide a non-empty answer                                                                            │
│  - If unsure, say:                                                                                              │
│    "The answer is not available in the knowledge base."                                                         │
│                                                                                                                 │
│  Return ONLY JSON:                                                                                              │
│  {                                                                                                              │
│    "question": "What does light reaction produce?",                                                             │
│    "answer": "..."                                                                                              │
│  }                                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Question: Capital of France?                                                                                   │
│                                                                                                                 │
│  Original Answer:                                                                                               │
│  The answer is not available in the knowledge base.                                                             │
│                                                                                                                 │
│  Retrieved Context:                                                                                             │
│  bisphosphate (RuBP). Through a series of reactions, the fixed carbon is processed and eventually contributes   │
│  to the formation of glucose and other carbohydrates.                                                           │
│                                                                                                                 │
│  from sunlight is used to transform carbon dioxide from the air and water from the soil into glucose, a sugar   │
│  that stores chemical energy. Oxygen is released as a byproduct of this process.                                │
│                                                                                                                 │
│  Evaluator Feedback:                                                                                            │
│  ['Out-of-scope question (adversarial)']                                                                        │
│                                                                                                                 │
│  Rewrite the answer so that:                                                                                    │
│  - It fixes ALL issues mentioned above                                                                          │
│  - It uses ONLY the retrieved context                                                                           │
│  - It does NOT hallucinate                                                                                      │
│  - If answer is not in context, clearly say:                                                                    │
│    "The answer is not available in the knowledge base."                                                         │
│                                                                                                                 │
│  Return JSON:                                                                                                   │
│  {                                                                                                              │
│    "revised_answer": "..."                                                                                      │
│  }                                                                                                              │
│                                                                                                                 │
│  Agent: Revisor                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 8e85df02-08be-4479-b972-8c7670265943                                                                       │
│  Final Output: {                                                                                                │
│    "revised_answer": "The answer is not available in the knowledge base."                                       │
│  }                                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

Output()

In [17]:
pd.DataFrame(results)

,question,initial_f,initial_r,verdict,final_f,final_r
0,What are the two stages?,1.0,1.00,PASS,1.0,1.0
1,What does light reaction produce?,1.0,1.00,PASS,1.0,1.0
2,What is RuBisCO?,0.5,0.75,FAIL,1.0,1.0
3,What affects photosynthesis rate?,1.0,1.00,PASS,1.0,1.0
4,What is photorespiration?,1.0,1.00,PASS,1.0,1.0
5,Who discovered gravity?,0.7,0.40,FAIL,0.7,0.4
6,Capital of France?,0.7,0.40,FAIL,0.7,0.4


In [18]:
init = sum(r["verdict"]=="PASS" for r in results)/len(results)*100
final = sum(r["final_f"]>=0.7 and r["final_r"]>=0.7 for r in results)/len(results)*100

print("Initial:", init)
print("Final:", final)

Initial: 57.14285714285714
Final: 71.42857142857143


## Reflection

1. **What types of questions caused the most failures, and why?**  
Failures mainly occurred in two scenarios:  
- Questions with vague or incomplete initial answers (e.g., light reaction, RuBisCO), where the model lacked specificity  
- Adversarial questions (e.g., “Who discovered gravity?”), where the answer was not present in the knowledge base  

In adversarial cases, the system correctly responded that the information was unavailable, but evaluation metrics still assigned partial scores due to the answer being contextually safe.


2. **How effective was the revision step? Did it consistently improve scores?**  
The revision step was effective in improving answer quality. For example, initial vague answers were rewritten to include precise, context-grounded information, leading to higher faithfulness and relevancy scores.  

However, revision did not apply to adversarial questions, since no additional context was available. Overall, the revision step consistently improved scores for valid knowledge-based queries.


3. **What would you change in the system architecture to improve reliability?**  
To improve reliability, I would enhance the retrieval stage using hybrid search (BM25 + dense retrieval) and optimize chunking strategies for better context coverage.  

Additionally, introducing confidence estimation or uncertainty detection would help identify out-of-scope queries more robustly, preventing misleading high scores for adversarial inputs.


4. **How would you extend this system with TruLens for ongoing monitoring?**  
The system can be extended using TruLens by logging each query, retrieved context, model output, and evaluation scores.  

This would enable continuous monitoring of performance, detection of failure patterns, and tracking of model drift over time. It would also allow comparison of different prompt or retrieval strategies to improve system reliability.